# 01 — Data Download: ERCOT LMP Prices

**Goal:** Download raw hourly LMP prices for ERCOT hubs (and optionally resource nodes) from our cloud data pipeline.

**Why raw prices?** The pipeline also has forecast-compressed prices (P15-P85), but BESS arbitrage is driven by extreme prices. A $9k/MWh spike clipped to $95 would massively undercount revenue. Raw LMP preserves the full distribution.

**Locations:**
- 4 ERCOT settlement hubs: HB_HOUSTON, HB_NORTH, HB_SOUTH, HB_WEST
- Up to 6 resource nodes at real generation sites (for hub vs node comparison)
- Both RT (real-time) and DA (day-ahead) prices

In [1]:
import os
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import duckdb
from google.cloud import storage

# Paths
DATA_DIR = os.path.join(os.pardir, 'data')
PRICES_DIR = os.path.join(DATA_DIR, 'prices')
os.makedirs(PRICES_DIR, exist_ok=True)

# GCS setup
GCS_PROJECT = 'infrasure-model-gpr'
GCS_BUCKET = 'infrasure-model-gpr-data'
client = storage.Client(project=GCS_PROJECT)
bucket = client.bucket(GCS_BUCKET)

# ERCOT hubs to download
HUBS = ['HB_HOUSTON', 'HB_NORTH', 'HB_SOUTH', 'HB_WEST']
MARKETS = ['rt_hourly', 'da_hourly']  # Real-time + Day-ahead

print(f"GCS bucket: {GCS_BUCKET}")
print(f"Hubs: {HUBS}")
print(f"Markets: {MARKETS}")
print(f"Output dir: {os.path.abspath(PRICES_DIR)}")

GCS bucket: infrasure-model-gpr-data
Hubs: ['HB_HOUSTON', 'HB_NORTH', 'HB_SOUTH', 'HB_WEST']
Markets: ['rt_hourly', 'da_hourly']
Output dir: /Users/divy/code/personal/BESS_modo/data/prices


## Download ERCOT Hub Prices (RT + DA)

In [2]:
downloaded = []

for hub in HUBS:
    for market in MARKETS:
        gcs_path = f'lmp_prices/{hub}/{market}.parquet'
        local_path = os.path.join(PRICES_DIR, f'{hub}_{market}.parquet')
        
        if os.path.exists(local_path):
            print(f"  Already exists: {local_path}")
            downloaded.append((hub, market, local_path))
            continue
            
        blob = bucket.blob(gcs_path)
        blob.download_to_filename(local_path)
        size_kb = os.path.getsize(local_path) / 1024
        print(f"  Downloaded: {hub}/{market} → {size_kb:.0f} KB")
        downloaded.append((hub, market, local_path))

print(f"\nTotal files: {len(downloaded)}")

  Already exists: ../data/prices/HB_HOUSTON_rt_hourly.parquet
  Already exists: ../data/prices/HB_HOUSTON_da_hourly.parquet
  Already exists: ../data/prices/HB_NORTH_rt_hourly.parquet
  Already exists: ../data/prices/HB_NORTH_da_hourly.parquet
  Already exists: ../data/prices/HB_SOUTH_rt_hourly.parquet
  Already exists: ../data/prices/HB_SOUTH_da_hourly.parquet
  Already exists: ../data/prices/HB_WEST_rt_hourly.parquet
  Already exists: ../data/prices/HB_WEST_da_hourly.parquet

Total files: 8


## Validate Downloaded Data

In [3]:
con = duckdb.connect()

validation_rows = []
for hub, market, path in downloaded:
    result = con.execute(f"""
        SELECT 
            '{hub}' as hub,
            '{market}' as market,
            COUNT(*) as rows,
            MIN(datetime) as start_date,
            MAX(datetime) as end_date,
            ROUND(AVG(price), 2) as avg_price,
            ROUND(MIN(price), 2) as min_price,
            ROUND(MAX(price), 2) as max_price,
            SUM(CASE WHEN price IS NULL THEN 1 ELSE 0 END) as null_count
        FROM read_parquet('{path}')
    """).fetchone()
    validation_rows.append(result)

validation_df = pd.DataFrame(validation_rows, 
    columns=['hub', 'market', 'rows', 'start_date', 'end_date', 'avg_price', 'min_price', 'max_price', 'null_count'])
validation_df

,hub,market,rows,start_date,end_date,avg_price,min_price,max_price,null_count
0,HB_HOUSTON,rt_hourly,132169,2010-12-01 01:00:00-05:00,2025-12-29 01:00:00-05:00,42.08,-163.63,9062.54,0
1,HB_HOUSTON,da_hourly,132191,2010-12-01 02:00:00-05:00,2025-12-30 00:00:00-05:00,43.68,-0.10,8995.11,0
2,HB_NORTH,rt_hourly,130538,2010-12-01 01:00:00-05:00,2025-10-22 03:00:00-04:00,40.59,-228.74,9203.58,0
3,HB_NORTH,da_hourly,130558,2010-12-01 02:00:00-05:00,2025-10-23 00:00:00-04:00,42.29,-4.00,8998.99,0
4,HB_SOUTH,rt_hourly,130333,2010-12-01 01:00:00-05:00,2025-10-13 14:00:00-04:00,40.47,-169.94,9003.87,0
5,HB_SOUTH,da_hourly,130332,2010-12-01 02:00:00-05:00,2025-10-13 14:00:00-04:00,42.31,-27.74,8987.66,0
6,HB_WEST,rt_hourly,130265,2010-12-01 01:00:00-05:00,2025-10-10 18:00:00-04:00,39.19,-367.63,9199.05,0
7,HB_WEST,da_hourly,130294,2010-12-01 02:00:00-05:00,2025-10-12 00:00:00-04:00,41.00,-28.07,9000.01,0


## Peek at Schema (one file)

In [4]:
# Look at one file to understand schema
sample = pd.read_parquet(os.path.join(PRICES_DIR, 'HB_HOUSTON_rt_hourly.parquet'))
print(f"Shape: {sample.shape}")
print(f"Index: {sample.index.name}, dtype={sample.index.dtype}")
print(f"\nColumns: {list(sample.columns)}")
print(f"\nDtypes:\n{sample.dtypes}")
print(f"\nHead:")
sample.head(3)

Shape: (132169, 7)
Index: datetime, dtype=datetime64[ns, UTC]

Columns: ['market', 'location', 'datetime_local', 'price', 'interval_start_utc', 'interval_end_utc', 'spp']

Dtypes:
market                                         object
location                                       object
datetime_local        datetime64[ns, America/Chicago]
price                                         float64
interval_start_utc                             object
interval_end_utc                               object
spp                                           float64
dtype: object

Head:


,market,location,datetime_local,price,interval_start_utc,interval_end_utc,spp
datetime,,,,,,,
2010-12-01 06:00:00+00:00,REAL_TIME_HOURLY,HB_HOUSTON,2010-12-01 00:00:00-06:00,24.8500,2010-12-01T05:00:00+00:00,2010-12-01T06:00:00+00:00,24.8500
2010-12-01 07:00:00+00:00,REAL_TIME_HOURLY,HB_HOUSTON,2010-12-01 01:00:00-06:00,23.4750,2010-12-01T06:00:00+00:00,2010-12-01T07:00:00+00:00,23.4750
2010-12-01 08:00:00+00:00,REAL_TIME_HOURLY,HB_HOUSTON,2010-12-01 02:00:00-06:00,21.4675,2010-12-01T07:00:00+00:00,2010-12-01T08:00:00+00:00,21.4675


## ERCOT BESS Fleet Summary

In [5]:
bess = pd.read_parquet(os.path.join(DATA_DIR, 'bess_enriched.parquet'))
ercot_op = bess[(bess['Balancing Authority Code'] == 'ERCO') & (bess['Status'] == '(OP) Operating')]

# Convert energy capacity to numeric (some values are strings)
ercot_op = ercot_op.copy()
ercot_op['Nameplate Energy Capacity (MWh)'] = pd.to_numeric(ercot_op['Nameplate Energy Capacity (MWh)'], errors='coerce')

print(f"ERCOT Operating BESS: {len(ercot_op)} units")
print(f"Total Capacity: {ercot_op['Nameplate Capacity (MW)'].sum():,.0f} MW")
print(f"Total Energy: {ercot_op['Nameplate Energy Capacity (MWh)'].sum():,.0f} MWh")

# Top 10 by capacity
top10 = ercot_op.nlargest(10, 'Nameplate Capacity (MW)')[
    ['Plant Name', 'Nameplate Capacity (MW)', 'Nameplate Energy Capacity (MWh)', 
     'RTO/ISO LMP Node Designation', 'Plant State', 'County']
].reset_index(drop=True)
top10.index += 1
top10

ERCOT Operating BESS: 155 units
Total Capacity: 10,105 MW
Total Energy: 15,355 MWh


,Plant Name,Nameplate Capacity (MW),Nameplate Energy Capacity (MWh),RTO/ISO LMP Node Designation,Plant State,County
1,SMT Ironman Battery Storage Plant,304.0,440.8,COSTAL,TX,Brazoria
2,Rodeo Ranch Energy Storage,300.0,600.0,None,TX,Reeves
3,DeCordova Steam Electric Station,269.1,263.0,None,TX,Hood
4,Five Wells Solar Center - Hybrid,262.5,250.0,None,TX,Bell
5,Anole Energy Storage,240.0,480.0,TRICRN1_5,TX,Dallas
6,Libra Storage,236.3,236.3,None,TX,Guadalupe
7,Callisto I Energy Center,223.4,200.0,None,TX,Harris
8,Ebony Energy Storage,207.9,400.0,None,TX,Comal
9,Citadel Battery Storage Plant,204.0,298.0,HOUSTON,TX,Harris
10,Crossett Power Management LLC,203.0,400.0,None,TX,Crane


In [6]:
print("✓ Data download complete. Files in data/prices/:")
for f in sorted(os.listdir(PRICES_DIR)):
    if f.endswith('.parquet'):
        size = os.path.getsize(os.path.join(PRICES_DIR, f)) / 1024
        print(f"  {f} ({size:.0f} KB)")

✓ Data download complete. Files in data/prices/:
  HB_HOUSTON_da_hourly.parquet (4219 KB)
  HB_HOUSTON_rt_hourly.parquet (4402 KB)
  HB_NORTH_da_hourly.parquet (5422 KB)
  HB_NORTH_rt_hourly.parquet (5609 KB)
  HB_SOUTH_da_hourly.parquet (5416 KB)
  HB_SOUTH_rt_hourly.parquet (5602 KB)
  HB_WEST_da_hourly.parquet (5425 KB)
  HB_WEST_rt_hourly.parquet (5640 KB)


---

## Part 2: Site-Specific Data (Nodal Prices, Generation, Revenue)

We have 6 ERCOT assets in the asset registry with **resource node pricing** — real solar/wind sites where we can model co-located BESS using actual nodal LMP instead of hub prices. This enables basis risk analysis (hub vs node revenue).

In [7]:
# Master site configuration — 6 ERCOT assets with resource nodes in GCS
SITES = {
    'lamesa_solar':           {'type': 'solar', 'mw': 102.0,  'node': 'LAMESASLR_G',   'hub': 'HB_WEST'},
    'misae_solar':            {'type': 'solar', 'mw': 240.0,  'node': 'MISAE_GEN_RN',  'hub': 'HB_WEST'},
    'longhorn_wind':          {'type': 'wind',  'mw': 200.0,  'node': 'LHORN_N_U1_2',  'hub': 'HB_WEST'},
    'panther_creek_wind_i':   {'type': 'wind',  'mw': 142.5,  'node': 'PC_NORTH_1',    'hub': 'HB_WEST'},
    'spinning_spur_wind_iii': {'type': 'wind',  'mw': 194.0,  'node': 'SSPURT_WIND1',  'hub': 'HB_WEST'},
    'stanton_wind_energy':    {'type': 'wind',  'mw': 120.0,  'node': 'SWEC_G1',       'hub': 'HB_WEST'},
}

NODES = [s['node'] for s in SITES.values()]
GEN_DIR = os.path.join(DATA_DIR, 'generation')
REV_DIR = os.path.join(DATA_DIR, 'revenue')
os.makedirs(GEN_DIR, exist_ok=True)

print(f"Sites: {len(SITES)}")
for slug, info in SITES.items():
    print(f"  {slug}: {info['type']} {info['mw']} MW @ {info['node']} (hub: {info['hub']})")

Sites: 6
  lamesa_solar: solar 102.0 MW @ LAMESASLR_G (hub: HB_WEST)
  misae_solar: solar 240.0 MW @ MISAE_GEN_RN (hub: HB_WEST)
  longhorn_wind: wind 200.0 MW @ LHORN_N_U1_2 (hub: HB_WEST)
  panther_creek_wind_i: wind 142.5 MW @ PC_NORTH_1 (hub: HB_WEST)
  spinning_spur_wind_iii: wind 194.0 MW @ SSPURT_WIND1 (hub: HB_WEST)
  stanton_wind_energy: wind 120.0 MW @ SWEC_G1 (hub: HB_WEST)


### Download Resource Node LMP Prices

In [8]:
node_downloaded = []

for node in NODES:
    for market in MARKETS:
        gcs_path = f'lmp_prices/{node}/{market}.parquet'
        local_path = os.path.join(PRICES_DIR, f'{node}_{market}.parquet')
        
        if os.path.exists(local_path):
            print(f"  Already exists: {node}/{market}")
            node_downloaded.append((node, market, local_path))
            continue
        
        blob = bucket.blob(gcs_path)
        if not blob.exists():
            print(f"  MISSING: {gcs_path}")
            continue
            
        blob.download_to_filename(local_path)
        size_kb = os.path.getsize(local_path) / 1024
        print(f"  Downloaded: {node}/{market} → {size_kb:.0f} KB")
        node_downloaded.append((node, market, local_path))

print(f"\nNode price files: {len(node_downloaded)}")

  Downloaded: LAMESASLR_G/rt_hourly → 3560 KB


  Downloaded: LAMESASLR_G/da_hourly → 5252 KB


  Downloaded: MISAE_GEN_RN/rt_hourly → 2498 KB


  Downloaded: MISAE_GEN_RN/da_hourly → 2348 KB


  Downloaded: LHORN_N_U1_2/rt_hourly → 4399 KB


  Downloaded: LHORN_N_U1_2/da_hourly → 4194 KB


  Downloaded: PC_NORTH_1/rt_hourly → 5694 KB


  Downloaded: PC_NORTH_1/da_hourly → 5435 KB
  MISSING: lmp_prices/SSPURT_WIND1/rt_hourly.parquet
  MISSING: lmp_prices/SSPURT_WIND1/da_hourly.parquet


  Downloaded: SWEC_G1/rt_hourly → 5717 KB


  Downloaded: SWEC_G1/da_hourly → 5453 KB

Node price files: 10


### Download Generation Data for 6 Sites

In [9]:
gen_downloaded = []

for slug, info in SITES.items():
    gen_subdir = 'solar_gen_data' if info['type'] == 'solar' else 'wind_gen_data'
    gcs_path = f'generation_data/{gen_subdir}/{slug}_generation.parquet'
    local_path = os.path.join(GEN_DIR, f'{slug}_generation.parquet')
    
    if os.path.exists(local_path):
        print(f"  Already exists: {slug}")
        gen_downloaded.append((slug, local_path))
        continue
    
    blob = bucket.blob(gcs_path)
    if not blob.exists():
        print(f"  MISSING: {gcs_path}")
        continue
    
    blob.download_to_filename(local_path)
    size_kb = os.path.getsize(local_path) / 1024
    print(f"  Downloaded: {slug} → {size_kb:.0f} KB")
    gen_downloaded.append((slug, local_path))

print(f"\nGeneration files: {len(gen_downloaded)}")

  Downloaded: lamesa_solar → 47952 KB


  Downloaded: misae_solar → 47744 KB


  Downloaded: longhorn_wind → 32053 KB
  MISSING: generation_data/wind_gen_data/panther_creek_wind_i_generation.parquet


  Downloaded: spinning_spur_wind_iii → 31947 KB
  MISSING: generation_data/wind_gen_data/stanton_wind_energy_generation.parquet

Generation files: 4


### Download Historical Revenue Data

In [10]:
rev_downloaded = []

for slug in SITES:
    slug_rev_dir = os.path.join(REV_DIR, slug, 'hub')
    os.makedirs(slug_rev_dir, exist_ok=True)
    
    for market in ['da', 'rt']:
        gcs_path = f'revenue_data/{slug}/hub/{market}_historical.parquet'
        local_path = os.path.join(slug_rev_dir, f'{market}_historical.parquet')
        
        if os.path.exists(local_path):
            print(f"  Already exists: {slug}/hub/{market}")
            rev_downloaded.append((slug, market, local_path))
            continue
        
        blob = bucket.blob(gcs_path)
        if not blob.exists():
            print(f"  MISSING: {gcs_path}")
            continue
        
        blob.download_to_filename(local_path)
        size_kb = os.path.getsize(local_path) / 1024
        print(f"  Downloaded: {slug}/hub/{market} → {size_kb:.0f} KB")
        rev_downloaded.append((slug, market, local_path))

print(f"\nRevenue files: {len(rev_downloaded)}")

  Downloaded: lamesa_solar/hub/da → 9439 KB


  Downloaded: lamesa_solar/hub/rt → 9508 KB


  Downloaded: misae_solar/hub/da → 9408 KB


  Downloaded: misae_solar/hub/rt → 9479 KB


  Downloaded: longhorn_wind/hub/da → 11162 KB


  Downloaded: longhorn_wind/hub/rt → 11240 KB
  MISSING: revenue_data/panther_creek_wind_i/hub/da_historical.parquet


  MISSING: revenue_data/panther_creek_wind_i/hub/rt_historical.parquet


  Downloaded: spinning_spur_wind_iii/hub/da → 11116 KB


  Downloaded: spinning_spur_wind_iii/hub/rt → 11200 KB
  MISSING: revenue_data/stanton_wind_energy/hub/da_historical.parquet
  MISSING: revenue_data/stanton_wind_energy/hub/rt_historical.parquet

Revenue files: 8


### Download Asset Registry

In [11]:
registry_path = os.path.join(DATA_DIR, 'asset_registry.duckdb')
if not os.path.exists(registry_path):
    blob = bucket.blob('asset_registry.duckdb')
    blob.download_to_filename(registry_path)
    print(f"Downloaded asset_registry.duckdb ({os.path.getsize(registry_path)/1024/1024:.1f} MB)")
else:
    print(f"Already exists: asset_registry.duckdb ({os.path.getsize(registry_path)/1024/1024:.1f} MB)")

# Show ERCOT assets with pricing config
reg = duckdb.connect(registry_path, read_only=True)
ercot_assets = reg.execute("""
    SELECT a.asset_name, a.asset_type, a.state, 
           ap.resource_node, ap.hub_node, ap.price_strategy
    FROM asset a
    JOIN asset_price ap USING (asset_id)
    WHERE a.balancing_authority = 'ERCO'
    ORDER BY a.asset_type, a.asset_name
""").fetchdf()
reg.close()

print(f"\nERCOT assets in registry: {len(ercot_assets)}")
ercot_assets

Downloaded asset_registry.duckdb (18.8 MB)

ERCOT assets in registry: 1


,asset_name,asset_type,state,resource_node,hub_node,price_strategy
0,Ash Creek Solar,solar,TX,None,HB_NORTH,hub_only


### Validate Node Price Data

In [12]:
node_validation = []
for node, market, path in node_downloaded:
    result = con.execute(f"""
        SELECT 
            '{node}' as node,
            '{market}' as market,
            COUNT(*) as rows,
            MIN(datetime) as start_date,
            MAX(datetime) as end_date,
            ROUND(AVG(price), 2) as avg_price,
            ROUND(MIN(price), 2) as min_price,
            ROUND(MAX(price), 2) as max_price
        FROM read_parquet('{path}')
    """).fetchone()
    node_validation.append(result)

node_val_df = pd.DataFrame(node_validation,
    columns=['node', 'market', 'rows', 'start_date', 'end_date', 'avg_price', 'min_price', 'max_price'])
node_val_df

,node,market,rows,start_date,end_date,avg_price,min_price,max_price
0,LAMESASLR_G,rt_hourly,76840,2017-01-04 02:00:00-05:00,2025-10-10 18:00:00-04:00,49.54,-251.00,9198.81
1,LAMESASLR_G,da_hourly,130294,2010-12-01 02:00:00-05:00,2025-10-12 00:00:00-04:00,50.90,-225.16,9016.40
2,MISAE_GEN_RN,rt_hourly,50706,2019-12-31 20:00:00-05:00,2025-10-13 14:00:00-04:00,49.48,-114.58,9200.90
3,MISAE_GEN_RN,da_hourly,50706,2019-12-31 20:00:00-05:00,2025-10-13 14:00:00-04:00,51.71,-37.76,9000.93
4,LHORN_N_U1_2,rt_hourly,98173,2014-10-01 02:00:00-04:00,2025-12-12 13:00:00-05:00,36.28,-112.02,9200.75
5,LHORN_N_U1_2,da_hourly,98160,2014-10-02 02:00:00-04:00,2025-12-13 00:00:00-05:00,38.52,-39.12,9000.98
6,PC_NORTH_1,rt_hourly,130333,2010-12-01 01:00:00-05:00,2025-10-13 14:00:00-04:00,39.21,-251.00,9198.47
7,PC_NORTH_1,da_hourly,130332,2010-12-01 02:00:00-05:00,2025-10-13 14:00:00-04:00,41.24,-65.44,9006.26
8,SWEC_G1,rt_hourly,130334,2010-12-01 01:00:00-05:00,2025-10-13 15:00:00-04:00,44.72,-262.90,9198.46
9,SWEC_G1,da_hourly,130333,2010-12-01 02:00:00-05:00,2025-10-13 15:00:00-04:00,46.54,-34.62,9035.31


In [13]:
print("✓ All data downloaded. Summary:")
print(f"  Hub prices:   {len(downloaded)} files")
print(f"  Node prices:  {len(node_downloaded)} files")
print(f"  Generation:   {len(gen_downloaded)} files")
print(f"  Revenue:      {len(rev_downloaded)} files")
print(f"  Asset registry: {'✓' if os.path.exists(registry_path) else '✗'}")
print(f"\nTotal parquet files in data/prices/: {len([f for f in os.listdir(PRICES_DIR) if f.endswith('.parquet')])}")

✓ All data downloaded. Summary:
  Hub prices:   8 files
  Node prices:  10 files
  Generation:   4 files
  Revenue:      8 files
  Asset registry: ✓

Total parquet files in data/prices/: 18
